# Maintenance
旧 thread_id（`agent_id` ごとの世代交代で使われなくなった checkpoint）を手動で削除する。

In [ ]:
import itertools
import os

from sqlalchemy import delete, func, select

from assistant_agent.entities.checkpoint import (
    CheckpointBlobEntity,
    CheckpointEntity,
    CheckpointWriteEntity,
)
from assistant_agent.store import PostgresStoreConnector

KEEP_LATEST_N = 3  # agent_id ごとに残す世代数

engine = PostgresStoreConnector(os.environ["AA_PG_CONNECTION_STRING"]).get_engine()

## セル1: 削除対象を表示（確認用）

In [ ]:
agent_id_col = func.split_part(CheckpointEntity.thread_id, ":", 1)
async with engine.connect() as conn:
    stmt = (
        select(
            agent_id_col.label("agent_id"),
            CheckpointEntity.thread_id,
            func.max(CheckpointEntity.checkpoint_id).label("latest"),
        )
        .group_by(agent_id_col, CheckpointEntity.thread_id)
        .order_by(agent_id_col, func.max(CheckpointEntity.checkpoint_id).desc())
    )
    rows = (await conn.execute(stmt)).all()

stale_thread_ids = []
for agent_id, group in itertools.groupby(rows, key=lambda row: row.agent_id):
    stale_thread_ids += [row.thread_id for row in list(group)[KEEP_LATEST_N:]]
print(f"Deleting {len(stale_thread_ids)} thread(s): {stale_thread_ids}")

## セル2: 実際に削除する（セル1を確認してから実行）

In [ ]:
async with engine.begin() as conn:
    for entity in (CheckpointEntity, CheckpointBlobEntity, CheckpointWriteEntity):
        await conn.execute(delete(entity).where(entity.thread_id.in_(stale_thread_ids)))